In [1]:
!pwd

/nfs/roberts/project/pi_skr2/eng26/scmpra/tabula-rasa/notebooks/object_creation/orthos/seelig_ortho_testing


In [2]:
#imports

#from tensorzinb.tensorzinb import TensorZINB
import scMPRAforge as scm
from scMPRAforge.core import _smart_matrix, _mom_from_training_data, _matricies_to_order, _tensorzinb_fit

2026-01-28 14:46:32.197968: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
import pandas as pd
import numpy as np
import time
import pickle
from formulaic import Formula
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
#create dask cluster

from dask_jobqueue import SLURMCluster
from dask.distributed import Client

cluster=SLURMCluster(
    cores=2,#cores per slurm job
    memory="512G",#memory per slurm job
    processes=1,#dask workers per slurm job
    job_extra_directives=["-p week", 
        f"--job-name=seelig_fit_worker",
        f"--time=7-00:00:00",
        f"--output=worker_%j.out"]
)

cluster.scale(jobs=2)

client = Client(cluster,
        timeout=f"{5*60}s",   # Client <-> scheduler timeout 
        heartbeat_interval="20s"  # Worker heartbeat interval
    )

#from dask.distributed import Client, LocalCluster
#cluster=LocalCluster(memory_limit='8GB')
#client = Client(cluster)

In [7]:
print(client.dashboard_link, flush=True)

http://10.18.22.90:40717/status


# Describe with Ortho


In [ ]:
data_root="/nfs/roberts/project/pi_skr2/shared/tabula_data"
path=f"{data_root}/seelig"
name="ortho_seelig_v1"

import os
if os.path.isdir(path+"/"+name):
    print("[+] Model found. Loading...")
    primordial=scm.ortho.load(client,path,name)
    shendure=primordial.training_data
else:
    print("[+] Model not found. Creating...")

    #load data
    seelig = load_and_preprocess_data(data_root)
    print('data loaded', flush=True)

    primordial=scm.ortho()
    primordial.criss_cross(client=client,
                       dat=seelig)
    primordial.extract_params(client)
    primordial.save(path,name)


In [ ]:
# def single_model_fit(data, split, design_only=False):
#     data = data.data
#     levels=data[split].unique()
#     t = levels[0]

#     #smart matrix
#     t_future = client.submit(
#             _smart_matrix,
#             data=data[data[split]==t],
#             split=split
#         )
#     if design_only:
#         return {t:t_future}
#     print('smart matrix done', flush=True)
#     # mom for training data, matrices to order
#     if split=="cell_type":
#         init_method="pass"
#         init_vals=client.submit(_mom_from_training_data, 
#                 data=data,
#                 split="cell_type",
#                 subset=t,
#                 indicies=client.submit(_matricies_to_order, matricies=t_future)
#                 )
#         print('MoM done', flush=True)

#     else:
#             init_method="nb"
#             init_vals= None 

#     #tensorzinb fit
#     print('submited fitting', flush=True)
#     tzinb_futures = client.submit(
#                 _tensorzinb_fit,
#                 t_future,
#                 t,
#                 init_method=init_method,
#                 init_vals=init_vals
#             )

#     print('fitting done', flush=True)
#     return(scm.experiment_model(model={t:tzinb_futures},
#                                 split=split),
#             {t:t_future})



In [26]:
design = single_model_fit(seelig, 'cell_type', design_only=True)


In [27]:
design

{'reference': <Future: pending, key: _smart_matrix-20ae9fdaff70134132e5bf875de0a077>}

In [ ]:
model.save(f"{data_root}/seelig/ortho_test_seelig/demo_cell_type_0_model.pkl")


In [ ]:
print("finished!", flush=True)

finished!


In [ ]:
# model = scm.experiment_model()
model = scm.experiment_model.load(client, f"{data_root}/seelig/ortho_test_seelig/7_daycell_type_0_model.pkl")

In [21]:
model.model['K562'].result()


{'llf_total': -533754.7453260745,
 'llfs': array([-533754.74532607]),
 'aic_total': 1070067.490652149,
 'aics': array([1070067.49065215]),
 'df_model_total': 1279,
 'df': 1279,
 'weights': {'x_mu': array([[-7.489379 ],
         [ 1.9074049],
         [ 3.225586 ],
         ...,
         [ 2.3302062],
         [ 2.3302307],
         [ 1.735548 ]], dtype=float32),
  'x_pi': array([[-3.596906]], dtype=float32),
  'theta': array([[-3.7460117]], dtype=float32)},
 'cpu_time': 135560.03298139572,
 'num_sample': 6630264,
 'epochs': 2813,
 'loss_history': [0.002430702792480588,
  0.001861794968135655,
  0.0014631843660026789,
  0.0011384603567421436,
  0.0008568570483475924,
  0.0006045647314749658,
  0.0003733298508450389,
  0.00015837917453609407,
  -4.363779225968756e-05,
  -0.0002349857531953603,
  -0.00041717354906722903,
  -0.0005916005466133356,
  -0.0007591749890707433,
  -0.0009208260453306139,
  -0.0010769421933218837,
  -0.001228182460181415,
  -0.0013749623904004693,
  -0.0015176865

In [ ]:
# client.close()
# cluster.close()